# gammaConversionSi — ntuple exploration

Reads the `conversions` ntuple produced by the `gammaConversionSi` study and writes one PDF per
branch into `training/plots/`, in the ATLAS style. Nothing is drawn inside the notebook — every
figure is saved and closed.

One row per converted photon. Units are **MeV**, **mm** and **radians** throughout — see
`studies/gammaConversionSi/README.md` for the full column description.

Things worth checking on the plots:

- `eGamma` on a log x-axis is **flat from ~3 MeV up**, confirming the log-uniform gun. Only the
  lowest bin sits low (92 % of the plateau), because just above the 1.022 MeV threshold the mean
  free path is still comparable to the 10 m block and some photons leave without converting —
  the ntuple holds only *converted* photons. The shape is the gun spectrum times the conversion
  probability, and with 10 m of silicon that probability is ≥99 % everywhere above 3 MeV.
- `pathInBlock` falls steeply, but is **concave on a log y-axis rather than a straight line**.
  A single exponential would be straight; this run superposes many energies, each with its own
  mean free path. Use `config/mono1GeV.cfg` for the clean single-exponential version.
- `phiElectron` and `phiPositron` should be **flat over −π…π** — azimuthal symmetry.
- `isTriplet` should sit near **6.7 %** — the 1/(Z+1) triplet fraction for silicon.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np
import uproot

hep.style.use(hep.style.ATLAS)

# The ATLAS style sets mathtext.fontset="custom" and points every math font at
# TeX Gyre Heros, which has no sized-radical glyph. The \sqrt in the ATLAS
# `com` label then warns "No TeX to Unicode mapping for '\__radicalbig__'" and
# falls back to '?'. stixsans supplies the glyph; fontset stays "custom", so
# every other math character is still TeX Gyre Heros and the style is intact.
plt.rcParams["mathtext.fallback"] = "stixsans"

In [2]:
def find_repo_root(start=None):
    """Walk up from `start` until a directory containing .git is found.

    Keeps the notebook working whatever directory Jupyter was launched from,
    rather than hard-coding a relative path back to the build tree.
    """
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError(f"no .git found above {path}")


REPO = find_repo_root()

NTUPLE = REPO / "build/gammaConversionSi/ntuples/Si_2MeV-100GeV_10000000.root"
PLOTS = REPO / "studies/gammaConversionSi/training/plots"
ENTRY_STOP = None  # set e.g. 500_000 to iterate quickly on a subset
COM = "2MeV - 100GeV"

PLOTS.mkdir(parents=True, exist_ok=True)

if not NTUPLE.exists():
    raise FileNotFoundError(
        f"{NTUPLE} not found. Generate it from the repository root with:\n"
        f"    ./studies/gammaConversionSi/build.sh\n"
        f"    source install/geant4/bin/geant4.sh\n"
        f"    cd build/gammaConversionSi && ./gammaConversionSi config/default.cfg"
    )

print(f"repository : {REPO}")
print(f"ntuple     : {NTUPLE.relative_to(REPO)}  ({NTUPLE.stat().st_size / 1e6:.0f} MB)")
print(f"plots      : {PLOTS.relative_to(REPO)}")

repository : /Users/s2612909/Geant4-Fast-Parameterisation-Models
ntuple     : build/gammaConversionSi/ntuples/Si_2MeV-100GeV_10000000.root  (970 MB)
plots      : studies/gammaConversionSi/training/plots


In [3]:
tree = uproot.open(NTUPLE)["conversions"]

print(f"entries: {tree.num_entries:,}")
print("branches:")
for name, branch in tree.items():
    print(f"  {name:<16} {branch.interpretation}")

entries: 9,977,028.0
branches:
  eGamma           AsDtype('>f8')
  pathInBlock      AsDtype('>f8')
  eElectron        AsDtype('>f8')
  ePositron        AsDtype('>f8')
  thetaElectron    AsDtype('>f8')
  thetaPositron    AsDtype('>f8')
  phiElectron      AsDtype('>f8')
  phiPositron      AsDtype('>f8')
  openingAngle     AsDtype('>f8')
  zConv            AsDtype('>f8')
  eRecoil          AsDtype('>f8')
  isTriplet        AsDtype('>i4')


In [4]:
def plot_branch(column, xlabel, logy=False, logx=False, bins=100):
    """Histogram one branch and save it to PLOTS as a PDF.

    column  name of the branch to read
    xlabel  axis label, LaTeX allowed
    logy    logarithmic y-axis
    logx    logarithmic x-axis and log-spaced bins; several branches span
            5-10 orders of magnitude and are unreadable on a linear axis
    bins    bin count, or an explicit array of bin edges

    Returns the path written. The figure is closed rather than shown, so
    nothing is embedded in the notebook.
    """
    values = tree[column].array(entry_stop=ENTRY_STOP, library="np").astype(float)

    if logx:
        values = values[values > 0]  # log spacing needs a positive floor
        edges = np.logspace(np.log10(values.min()), np.log10(values.max()), bins + 1)
    elif np.ndim(bins):
        edges = np.asarray(bins)  # explicit edges, used for the isTriplet flag
    else:
        edges = np.linspace(values.min(), values.max(), bins + 1)

    counts, edges = np.histogram(values, bins=edges)

    fig, ax = plt.subplots()
    # yerr=True is mplhep's Garwood Poisson interval: asymmetric, and non-zero
    # for empty bins, rather than the sqrt(N) Gaussian approximation
    hep.histplot(counts, edges, ax=ax, yerr=True, histtype="step", color="black")

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Conversions")
    if logx:
        ax.set_xscale("log")

    # Headroom for the ATLAS label, which would otherwise sit on top of the
    # data for distributions that fill the frame
    peak = counts.max() + np.sqrt(counts.max())
    if logy:
        ax.set_yscale("log")
        floor = max(counts[counts > 0].min(), 1) if (counts > 0).any() else 1
        ax.set_ylim(0.5 * floor, peak * 10)
    else:
        ax.set_ylim(0, peak * 1.35)

    hep.atlas.label(text="Internal", com=COM, ax=ax)

    path = PLOTS / f"{column}.pdf"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path

In [5]:
BRANCHES = [
    ("eGamma",        r"$E_\gamma$ [MeV]",             dict(logx=True)),
    ("pathInBlock",   r"Path in block [mm]",           dict(logy=True)),
    ("eElectron",     r"$E_{e^-}$ [MeV]",              dict(logx=True)),
    ("ePositron",     r"$E_{e^+}$ [MeV]",              dict(logx=True)),
    ("thetaElectron", r"$\theta_{e^-}$ [rad]",         dict(logx=True, logy=True)),
    ("thetaPositron", r"$\theta_{e^+}$ [rad]",         dict(logx=True, logy=True)),
    ("phiElectron",   r"$\phi_{e^-}$ [rad]",           dict()),
    ("phiPositron",   r"$\phi_{e^+}$ [rad]",           dict()),
    ("openingAngle",  r"$\theta_{e^+e^-}$ [rad]",      dict(logx=True, logy=True)),
    ("zConv",         r"$z_{\mathrm{conv}}$ [mm]",     dict(logy=True)),
    ("eRecoil",       r"$E_{\mathrm{recoil}}$ [MeV]",  dict(logx=True, logy=True)),
    ("isTriplet",     "Triplet conversion",
     dict(logy=True, bins=np.array([-0.5, 0.5, 1.5]))),
]

for column, xlabel, kwargs in BRANCHES:
    path = plot_branch(column, xlabel, **kwargs)
    print(f"saved {path.relative_to(REPO)}")

saved studies/gammaConversionSi/training/plots/eGamma.pdf
saved studies/gammaConversionSi/training/plots/pathInBlock.pdf
saved studies/gammaConversionSi/training/plots/eElectron.pdf
saved studies/gammaConversionSi/training/plots/ePositron.pdf
saved studies/gammaConversionSi/training/plots/thetaElectron.pdf
saved studies/gammaConversionSi/training/plots/thetaPositron.pdf
saved studies/gammaConversionSi/training/plots/phiElectron.pdf
saved studies/gammaConversionSi/training/plots/phiPositron.pdf
saved studies/gammaConversionSi/training/plots/openingAngle.pdf
saved studies/gammaConversionSi/training/plots/zConv.pdf
saved studies/gammaConversionSi/training/plots/eRecoil.pdf
saved studies/gammaConversionSi/training/plots/isTriplet.pdf
